# Exercise 15: Decoherence and filamentation

**Block 5: Landau damping, conceptually**

**Learning objectives**
* Kick a Gaussian beam and watch its centroid oscillate: forever in a linear ring, but decaying within tens
  of turns when an octupole gives the particles different tunes (**decoherence**).
* See the beam turn into a spiral in phase space (**filamentation**) and its rms emittance grow, while each
  particle's own $\varepsilon_i$ stays (nearly) constant.
* Connect this to the two emittances of Exercise 6, and to the idea of Landau damping.

**Motivation.** When a beam is kicked (by an injection error, a magnet failure, or on purpose to measure the
tune) the whole beam oscillates together. Instruments see this *coherent* oscillation. With a tune spread the
signal disappears quickly, although no particle has lost any energy. Where did the oscillation go?

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, lattices as L, tracking as T, diagnostics as D, beams as B, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()
rng = np.random.default_rng(7)

## The setup

* A Hénon-like ring with $Q_0 = 0.28$ and a thin octupole at a place with $\beta$ = 50 m (octupoles are put
  where $\beta$ is large: their effect grows like $\beta^2$, Exercise 14).
* A matched Gaussian beam with $\varepsilon_\text{rms}$ = 1 mm mrad: $\sigma_x = \sqrt{\beta\varepsilon} \approx$ 7 mm.
* At turn 0 we kick the beam with a thin **dipole** (Exercise 1, `knl=[θ]`) so that its centre is displaced
  by one $\sigma$ in normalised phase space.

In [ ]:
BETA, EPS, Q0 = 50.0, 1e-6, 0.28
N = 20_000
beam0 = B.matched_gaussian(N, EPS, BETA, 0.0, rng=rng)
theta = np.sqrt(EPS / BETA)                       # = sigma_px, i.e. one sigma in normalised units
beam_kicked = E.thin_multipole(beam0, knl=[-theta])   # dipole kick: dpx = +theta
print(f"sigma_x = {np.sqrt(BETA * EPS) * MM:.2f} mm, kick = {theta * 1e6:.1f} urad")

## 1. Track and record

Storing every turn of 20 000 particles would need a lot of memory, so we record only what we need.

**Task.** Complete `track_and_monitor`. For every turn record the centroid ⟨$x$⟩, the rms emittance
(`D.rms_emittance`) and the mean single-particle emittance ⟨\varepsilon_i⟩ (`D.cs_invariant` with $\beta$ = `BETA`,
$\alpha$ = 0, measured from the ring's axis); keep a copy of the bunch at the turns listed in `snapshots`.

In [ ]:
def track_and_monitor(bunch, ring, n_turns: int, snapshots=()):
    """Track turn by turn and return a dict with arrays 'xbar', 'eps_rms', 'eps_i_mean' (length n_turns + 1),
    'eps_i_first' and 'eps_i_last' (per particle), 'snap' (dict turn -> copy of the bunch) and 'n_lost'."""
    out = {"xbar": [], "eps_rms": [], "eps_i_mean": [], "snap": {}}
    p = bunch.copy()
    for turn in range(n_turns + 1):
        if turn > 0:
            p = ring.one_turn_map(p)
            p[:, ~(np.abs(p[0, :]) < 0.1)] = np.nan   # particles that run away are lost (NaN):
                                                   # use np.nanmean, which skips them
        # YOUR CODE HERE
        raise NotImplementedError("Replace this line with your code")
    out["eps_i_last"] = eps_i
    out["n_lost"] = int(np.isnan(p[0, :]).sum())
    return {k: (np.array(v) if isinstance(v, list) else v) for k, v in out.items()}

In [ ]:
ring_lin = L.HenonRing(Q0, 0.31, BETA, 0.0, 10.0, 0.0)            # no octupole
mon = track_and_monitor(beam_kicked[:2000], ring_lin, 20)
assert mon["xbar"].shape == (21,) and np.allclose(mon["eps_rms"], mon["eps_rms"][0])
print("GOOD! track_and_monitor works")

**Predict first.** Sketch the centroid ⟨$x$⟩ vs turn for (a) the linear ring and (b) the ring with an
octupole. And the rms emittance in both cases.

*Your prediction:*

In [ ]:
# (about 10 s)
K3L = 50.0
ring_oct = L.HenonRing(Q0, 0.31, BETA, 0.0, 10.0, 0.0, knl=[0, 0, 0, K3L])
snaps = (0, 10, 30, 100, 300, 1000)
n_turns = 1000
mon_lin = track_and_monitor(beam_kicked, ring_lin, n_turns)
mon_oct = track_and_monitor(beam_kicked, ring_oct, n_turns, snapshots=snaps)

## 2. Decoherence of the centroid

For a first-order detuning $\Delta Q = a J$ ($a = k_3L \beta^2 / 16\pi$, Exercise 14) and a Gaussian beam kicked by
$Z \sigma$, one can compute the centroid exactly (a Gaussian integral). Its amplitude decays as

$$A(n) = \frac{1}{1+\theta^2}\exp\!\left(-\frac{Z^2}{2}\,\frac{\theta^2}{1+\theta^2}\right),\qquad \theta = 2\pi\, a\, \varepsilon_{rms}\, n .$$

(The beam is not lost: the oscillation of the *centre* disappears because the particles get out of step.)

In [ ]:
sig = np.sqrt(BETA * EPS)
n = np.arange(n_turns + 1)
a = K3L * BETA**2 / (16 * np.pi)
th = 2 * np.pi * a * EPS * n
A_theory = np.exp(-0.5 * th**2 / (1 + th**2)) / (1 + th**2)

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(n, mon_lin["xbar"] / sig, color=COLORS["grey"], lw=0.8)
axs[0].set_ylabel(r"$\langle x\rangle / \sigma_x$"); axs[0].set_title("no octupole")
axs[1].plot(n, mon_oct["xbar"] / sig, color=COLORS["x"], lw=0.8, label="tracking")
axs[1].plot(n, A_theory, "k--", lw=1, label="theory envelope")
axs[1].plot(n, -A_theory, "k--", lw=1)
axs[1].set_ylabel(r"$\langle x\rangle / \sigma_x$"); axs[1].set_xlabel("turn")
axs[1].set_title(rf"octupole $k_3L$ = {K3L:.0f} $1/\text{{m}}^{{-3}}$"); axs[1].legend(loc="upper right")
axs[1].set_xlim(0, 400)
plt.show()

# the envelope of the tracked centroid: |complex normalised centroid|
env = np.abs(mon_oct["xbar"]) / sig
i_half = np.argmax(A_theory < 0.5)
print(f"centroid amplitude halves after ~{i_half} turns (theory)")
peak = lambda k: np.max(env[max(k - 3, 0):k + 4])        # local maximum over ~one oscillation
assert all(abs(peak(k) - A_theory[k]) < 0.06 for k in (20, 50, 100, 200)), "tracking should follow the envelope"
assert np.max(np.abs(mon_lin["xbar"][-50:])) > 0.95 * sig, "without octupole the oscillation never stops"
print("GOOD! Decoherence in tens of turns, as predicted")

## 3. Filamentation: where did the oscillation go?

In [ ]:
fig, axs = plt.subplots(1, len(snaps), figsize=(18, 3.6), sharex=True, sharey=True)
for ax, t in zip(axs, snaps):
    p = mon_oct["snap"][t]
    P.phase_space(ax, p[:4000, 0], p[:4000, 1], BETA, 0.0, s=0.5)
    ax.set_title(f"turn {t}")
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    if t != snaps[0]:
        ax.set_ylabel("")
fig.suptitle("Kicked beam with an octupole, normalised phase space")
plt.show()

The particles further out rotate faster (higher tune), so the blob is sheared into a **spiral**. After a few
hundred turns the arms of the spiral are so fine that the beam looks like a ring-shaped, centred
distribution: the centroid is zero, but the beam is **bigger** than before.

## 4. The two emittances again

**Predict first.** The kick displaced the beam by $Z\sigma$ with $Z = 1$. By how much do you expect the rms emittance
to grow once the beam has filamented? (Hint: Exercise 6, which emittance does the lattice "respect"?)

*Your prediction:*

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
axs[0].plot(n, mon_lin["eps_rms"] / EPS, color=COLORS["grey"], label=r"$\varepsilon_\text{rms}$, no octupole")
axs[0].plot(n, mon_oct["eps_rms"] / EPS, color=COLORS["x"], label=r"$\varepsilon_\text{rms}$, with octupole")
axs[0].plot(n, mon_oct["eps_i_mean"] / 2 / EPS, color=COLORS["accent"], lw=2, label=r"$\langle\varepsilon_i\rangle/2$, with octupole")
axs[0].axhline(1 + 0.5, color="k", ls=":", lw=1, label=r"$\varepsilon_0(1 + Z^2/2)$")
axs[0].set_xlabel("turn"); axs[0].set_ylabel(r"emittance / $\varepsilon_0$"); axs[0].legend(fontsize=8)
ratio = mon_oct["eps_i_last"] / mon_oct["eps_i_first"]
axs[1].hist(ratio, bins=np.linspace(0.8, 1.2, 81), color=COLORS["x"], alpha=0.7)
axs[1].set_xlabel(r"$\varepsilon_i$(turn 1000) / $\varepsilon_i$(turn 0)"); axs[1].set_ylabel("particles")
axs[1].set_title(r"individual particles keep their $\varepsilon_i$")
plt.show()

final = mon_oct["eps_rms"][-1] / EPS
half_mean = mon_oct["eps_i_mean"] / 2 / EPS
print(f"eps_rms: {mon_oct['eps_rms'][0] / EPS:.3f} -> {final:.3f} eps0;  <eps_i>/2: {half_mean[0]:.3f} -> {half_mean[-1]:.3f} eps0")
print(f"individual eps_i changed by {np.nanmedian(np.abs(ratio - 1)) * 100:.1f} % (median); "
      f"particles lost: {mon_oct['n_lost']}")
assert abs(final - 1.5) < 0.05 and abs(half_mean[0] - 1.5) < 0.05 and abs(half_mean[-1] - half_mean[0]) < 0.02
assert np.nanmedian(np.abs(ratio - 1)) < 0.05 and mon_oct["n_lost"] == 0

Here is the connection to Exercise 6:

* After the kick, the beam is a perfectly fine Gaussian with $\sqrt{\text{det} \Sigma} = \varepsilon_0$, but it is not centred: seen by the
  lattice it is **mismatched**. Its ⟨$\varepsilon_i$⟩/2 (measured from the ring axis) is $\varepsilon_0(1 + Z^2/2) = 1.5\varepsilon_0$.
* In a linear ring both numbers would stay as they are forever (grey line).
* With a tune spread, the lattice "mixes" the beam: each particle keeps its own $\varepsilon_i$ (the histogram is
  narrow; the small width comes from the octupole slightly distorting the circles), but the phases get
  scrambled. The rms emittance grows until it reaches ⟨$\varepsilon_i$⟩/2 = 1.5 $\varepsilon_0$, the value the lattice "sees".

The same happens to a beam injected with the wrong $\beta$ (Exercise 6, section 4): it filaments until
$\varepsilon_\text{rms}$ = ⟨$\varepsilon_i$⟩/2 = $B \varepsilon_0$.

In [ ]:
# (about 5 s) a mismatched beam as in Exercise 6, now in a ring with an octupole
B_mis = 2.0
Sigma_mis = EPS * np.array([[B_mis * BETA, 0], [0, 1 / (B_mis * BETA)]])
beam_mis = np.zeros((4, N)); beam_mis[:2] = B.gaussian_from_sigma(Sigma_mis, N, rng=rng)
mon_mis = track_and_monitor(beam_mis, ring_oct, 2000)
Bmag = 0.5 * (B_mis + 1 / B_mis)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(mon_mis["eps_rms"] / EPS, color=COLORS["x"], label=r"$\varepsilon_{rms}$")
ax.plot(mon_mis["eps_i_mean"] / 2 / EPS, color=COLORS["accent"], label=r"$\langle\varepsilon_i\rangle/2$")
ax.axhline(Bmag, color="k", ls=":", label=f"B = {Bmag:.2f}")
ax.set_xlabel("turn"); ax.set_ylabel(r"emittance / $\varepsilon_0$"); ax.legend()
ax.set_title(rf"beta-mismatched beam ($\beta_\text{{beam}} = {B_mis} \beta$) filaments to $B\varepsilon_0$")
plt.show()
print(f"after 2000 turns: eps_rms = {mon_mis['eps_rms'][-1] / EPS:.3f} eps0, "
      f"<eps_i>/2 = {mon_mis['eps_i_mean'][-1] / 2 / EPS:.3f} eps0 (B = {Bmag:.3f} for an infinite sample), "
      f"particles lost: {mon_mis['n_lost']}")
assert abs(mon_mis["eps_rms"][-1] / (mon_mis["eps_i_mean"][-1] / 2) - 1) < 0.02
assert abs(mon_mis["eps_i_mean"][0] / 2 / EPS - Bmag) < 0.03 * Bmag

## 5. Landau damping, conceptually

What we saw is **decoherence**: a coherent oscillation disappears because the particles have different
frequencies, without any friction. Nothing is lost and nothing is dissipated: the "information" is hidden in
the fine spiral structure, and the price is a larger rms emittance.

Real beams can also be driven *coherently* by their own electromagnetic fields (wakefields): the beam
leaves fields in the vacuum chamber that kick the beam on later turns. This can make a small coherent
oscillation *grow*: an **instability**. If the particles have a spread of tunes, the coherent motion keeps
dissolving into incoherent motion before it can grow. This stabilising effect is called **Landau damping**,
and it is the main reason why the LHC has hundreds of octupole magnets ("Landau octupoles"): they create a
controlled tune spread.

## Reflection questions

1. In the linear ring the centroid oscillation never decays. Is the beam then "stable"?
2. Why does the emittance growth after a kick not depend on the octupole strength (only how fast it
   happens)?
3. Why is it important to inject a beam on the right orbit and with the right $\beta$, even in a ring with very
   good vacuum and no losses?